# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).

**Nota de elaboración:** Las celdas Markdown cuentan con apoyo de inteligencia artificial generativa. Los ejemplos tienen fines educativos y demostrativos. El autor es responsable de su revisión, adaptación y validación en Microsoft Fabric.

# NB_04 — Generación de resúmenes con `ai.summarize`
**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo:** `gpt-5-mini`

## Objetivos
- Resumir una columna de texto y comparar con un resumen ejecutivo basado en múltiples columnas.
- Validar errores, longitud y fidelidad de los resúmenes.
- Auditar tokens y CU-segundos por ejecución.

**Referencias oficiales:** [Summarize](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/summarize) · [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration) · [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).

## 1. Requisitos

Capacidad Fabric compatible, AI Functions habilitadas por la administración y sesión PySpark con SynapseML. Para Runtime 2.0 (Spark 4.1), revisar la configuración temporal indicada en Microsoft Learn. Adjuntar un Lakehouse solo si se desea persistir. Usar datos ficticios: el texto se procesa mediante el endpoint del modelo. Cada llamada puede consumir capacidad.

## 2. Configuración del modelo

Configuramos el deployment generativo de AI Functions para la sesión actual. El modelo configurado debe contrastarse con el modelo reportado en las estadísticas. Las llamadas pueden generar consumo real de capacidad.

In [ ]:
from synapse.ml.spark import aifunc
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"
aifunc.default_conf.set_deployment_name(MODEL_NAME)
print(aifunc.default_conf.get_deployment_name())

## 3. Referencia del método publicado por Microsoft

[Use ai.summarize with PySpark — Microsoft Learn](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/summarize)

### Sintaxis oficial (referencia, no ejecutable)

```python
# Una columna
df.ai.summarize(input_col="text", output_col="summaries")

# Todas las columnas
df.ai.summarize(output_col="summaries")
```

### Parámetros

| Parámetro | Tipo | Requerido | Comportamiento si se omite |
|---|---|---|---|
| `input_col` | str | No | Considera todas las columnas de cada fila |
| `instructions` | str | No | Sin instrucciones adicionales del usuario |
| `output_col` | str | No | Nombre generado automáticamente |
| `error_col` | str | No | Nombre generado automáticamente |

**Retorno:** nuevo DataFrame con un resumen por fila. Para una columna de texto, una entrada nula devuelve un resultado nulo. Los errores se registran en la columna correspondiente.

### Extensión multimodal

Microsoft documenta `input_col_type="path"` para una columna y `col_types={"file_path": "path"}` para el modo de múltiples columnas. Consultar los requisitos antes de habilitar estas modalidades.

Los ejemplos siguientes son propios de esta serie. El resumen ejecutivo utiliza instrucciones en español y contexto operacional ficticio. No equivale a una agregación de todas las filas del DataFrame.


## 4. Dataset ficticio
Cada fila representa un incidente o solicitud. Los datos no corresponden a clientes reales. Se incluyen los campos de contexto para demostrar la diferencia entre resumir una columna y resumir toda la fila.

In [ ]:
data = [
    (1, "Incidente", "Alta", "Resuelto", "Durante la carga nocturna, el pipeline de ventas falló por una conexión expirada. El equipo renovó las credenciales, reejecutó el proceso y verificó que los registros llegaron completos al destino."),
    (2, "Rendimiento", "Media", "En análisis", "El reporte comercial demora más de lo habitual. Se detectaron consultas con filtros poco selectivos. El equipo revisará el modelo semántico y realizará pruebas antes de aplicar cambios."),
    (3, "Solicitud", "Baja", "Pendiente", "El área financiera solicita un tablero que consolide ingresos y gastos mensuales. Se requiere definir los indicadores y validar las fuentes con los responsables del negocio."),
    (4, "Calidad", "Alta", "Mitigado", "Se identificaron registros duplicados en una tabla de pedidos. Se aisló la carga afectada y se aplicó una deduplicación temporal. Falta corregir la lógica de origen."),
    (5, "Disponibilidad", "Media", "Resuelto", "Una actualización del servicio dejó temporalmente indisponible el acceso a un conjunto de datos. El servicio se restauró y las pruebas confirmaron el acceso normal."),
    (6, "Soporte", "Baja", "Pendiente", None)
]
df = spark.createDataFrame(data, ["id", "tipo", "prioridad", "estado", "descripcion"])
display(df)

## 5. Resumen de una columna
`input_col` selecciona el texto; `output_col` y `error_col` identifican las nuevas columnas. Un texto nulo produce un resultado nulo. Spark evalúa bajo demanda: materializamos en caché para reducir nuevas inferencias, aunque una pérdida de caché puede provocar recomputación.

In [ ]:
run_id_default = str(uuid.uuid4())
resultado = df.ai.summarize(input_col="descripcion", output_col="resumen", error_col="error_resumen")
resultado.cache()
resultado.count()
display(resultado)

## 6. Validación de los resultados
Comparamos caracteres como indicador descriptivo de compresión (no equivale a tokens). Una reducción de longitud no garantiza fidelidad. Revisar manualmente que se preserven hechos, estado y acciones pendientes.

In [ ]:
resultado.printSchema()
comparacion = resultado.withColumn("caracteres_original", F.length("descripcion")).withColumn("caracteres_resumen", F.length("resumen"))
comparacion = comparacion.withColumn("reduccion_pct", F.when(F.col("caracteres_original") > 0, F.round(100 * (1 - F.col("caracteres_resumen") / F.col("caracteres_original")), 2)))
display(comparacion)
display(resultado.filter(F.col("error_resumen").isNotNull() | (F.col("descripcion").isNotNull() & F.col("resumen").isNull())))

## 7. Consultar estadísticas con `ai.stats`

Una vez materializado el resultado, consultamos el DataFrame de estadísticas. Inspeccionar el esquema y la granularidad antes de interpretar los datos. Cada llamada se registra de forma independiente.

| Métrica | Interpretación |
|---|---|
| `input_tokens` | Entrada total |
| `cached_tokens` | Subconjunto de entrada servido desde caché |
| `output_tokens` | Salida total |
| `reasoning_tokens` | Razonamiento, informativo; no sumar nuevamente a salida si ya está incluido |
| `num_successful` | Filas procesadas exitosamente |
| `num_exceptions` | Excepciones reportadas |

[Configuración y estadísticas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).

In [ ]:
def read_stats(result):
    stats_df = result.ai.stats
    display(stats_df)
    rows = stats_df.collect()
    if len(rows) != 1:
        raise RuntimeError("Revisar el esquema y la granularidad de ai.stats antes de estimar CU.")
    return rows[0].asDict(recursive=True)

stats_default = read_stats(resultado)
for key in ("input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens"):
    print(f"{key}: {stats_default.get(key)}")

## 8. Estimar el consumo de capacidad

Tarifas de referencia de la serie para `gpt-5-mini` integrado:

| Tipo de token | CU-s por 1.000 tokens |
|---|---:|
| Entrada sin caché | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

$$CU_s = \frac{(I-C)\times8.40+C\times0.84+O\times67.23}{1000}$$

I: entrada total; C: entrada en caché; O: salida. Ejemplo ilustrativo: 1.000 tokens de entrada, 200 en caché y 100 de salida equivalen a 13,611 CU-s. No incluye Spark. Verificar modelo y tarifas vigentes en [Microsoft Learn — Billing](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Si faltan métricas, el código detiene el cálculo en lugar de inventar valores.

In [ ]:
def estimate_cu(stats):
    i, c, o = (stats.get(k) for k in ("input_tokens", "cached_tokens", "output_tokens"))
    if any(v is None for v in (i, c, o)):
        raise ValueError("Métricas de tokens incompletas; no es posible estimar CU.")
    i, c, o = int(i), int(c), int(o)
    if not 0 <= c <= i or o < 0:
        raise ValueError("Métricas inconsistentes")
    return ((i-c)*8.40 + c*0.84 + o*67.23)/1000

print("CU-segundos estimados:", estimate_cu(stats_default))

## 9. Resumen ejecutivo de múltiples columnas
Al omitir `input_col`, se genera un resumen por fila considerando todas las columnas del DataFrame seleccionado, no un resumen global del dataset. `instructions` orienta el idioma, extensión, audiencia y foco. Conservamos el identificador para trazabilidad y seleccionamos solo filas con descripción para comparar ambas modalidades. Esta llamada genera consumo adicional.

In [ ]:
df_ejecutivo = df.filter(F.col("descripcion").isNotNull()).select("id", "tipo", "prioridad", "estado", "descripcion")
# El identificador se mantiene para trazabilidad y se pide no incluirlo en el texto.
run_id_custom = str(uuid.uuid4())
resultado_custom = df_ejecutivo.ai.summarize(
    output_col="resumen_ejecutivo", error_col="error_ejecutivo",
    instructions="Redacta en español un resumen ejecutivo de máximo 40 palabras. Incluye prioridad, estado, impacto y acciones pendientes si constan en los datos. No incluyas el identificador ni inventes información."
)
resultado_custom.cache()
resultado_custom.count()
display(resultado_custom)
stats_custom = read_stats(resultado_custom)
print("CU-segundos estimados:", estimate_cu(stats_custom))

## 10. Comparar modalidades
La primera llamada utiliza únicamente la descripción; la segunda incorpora contexto adicional e instrucciones. No es una comparación aislada del efecto de las instrucciones. El resumen ejecutivo debe contrastarse con los campos de origen.

In [ ]:
display(resultado.select("id", "descripcion", "resumen").join(resultado_custom.select("id", "resumen_ejecutivo"), "id", "left"))

## 11. Auditoría de ambas llamadas
Se conserva el esquema de la serie. Una entrada nula no se considera error si devuelve un resultado nulo sin error. El estado técnico no mide la exactitud semántica.

In [ ]:
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])


def audit_row(run_id, stats, result, output, error):
    total = result.count()
    invalid = result.filter((F.col("descripcion").isNotNull() & F.col(output).isNull()) | F.col(error).isNotNull()).count()
    def metric(k):
        return int(stats[k]) if stats.get(k) is not None else None
    return (run_id, datetime.now(timezone.utc).isoformat(),
            "NB_04_AI_SUMMARIZE", "ai.summarize", MODEL_NAME,
            str(stats["model"]) if stats.get("model") is not None else None,
            total, metric("num_successful"), metric("num_exceptions"),
            metric("num_unevaluated"), metric("num_harmful"),
            metric("input_tokens"), metric("cached_tokens"), metric("output_tokens"),
            metric("reasoning_tokens"), float(estimate_cu(stats)),
            str(stats["client_type"]) if stats.get("client_type") is not None else None,
            "SUCCESS" if invalid == 0 else "PARTIAL" if invalid < total else "INCOMPLETE")

audit_df = spark.createDataFrame([
    audit_row(run_id_default, stats_default, resultado, "resumen", "error_resumen"),
    audit_row(run_id_custom, stats_custom, resultado_custom, "resumen_ejecutivo", "error_ejecutivo")
], audit_schema)
display(audit_df)
display(audit_df.agg(F.sum("estimated_cu_seconds").alias("total_cu_seconds")))
display(audit_df.agg(F.sum("input_tokens").alias("total_input_tokens"), F.sum("cached_tokens").alias("total_cached_tokens"), F.sum("output_tokens").alias("total_output_tokens")))

## 12. Persistencia opcional en Delta

Se mantiene el esquema de auditoría de la serie. Activar únicamente con un Lakehouse adjunto y ajustar el nombre si no utiliza schemas. Reejecutar la escritura en modo append puede duplicar registros.

In [ ]:
PERSIST_AUDIT = False
TABLE_NAME = "dbo.audit_ai_functions"
if PERSIST_AUDIT:
    audit_df.write.format("delta").mode("append").saveAsTable(TABLE_NAME)

## 13. Extensión multimodal (opcional)
Microsoft documenta el uso de `input_col_type="path"` para resumir archivos compatibles, como PDF o imágenes, mediante una columna con rutas. Revisar requisitos y disponibilidad en la [documentación multimodal](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). El siguiente bloque no ejecuta inferencias por defecto; si se habilita, su consumo debe auditarse por separado.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    file_path = "Files/ejemplo.pdf"  # Ajustar a un archivo compatible y accesible.
    archivos = spark.createDataFrame([(file_path,)], ["ruta"])
    resumen_archivo = archivos.ai.summarize(input_col="ruta", input_col_type="path", output_col="resumen", error_col="error", instructions="Resume en español los hechos principales del documento.")
    resumen_archivo.cache()
    resumen_archivo.count()
    display(resumen_archivo)
    stats_archivo = read_stats(resumen_archivo)
    print("CU-segundos estimados (no incluidos en auditoría principal):", estimate_cu(stats_archivo))
    resumen_archivo.unpersist()

## 14. Liberar caché

In [ ]:
resultado.unpersist()
resultado_custom.unpersist()

## 15. Conclusiones
`ai.summarize` permite condensar texto y contextualizar resúmenes mediante instrucciones. Revisar resultados frente a los datos originales. El cálculo de CU es una estimación del modelo integrado, no del consumo completo de Spark. Validar en Fabric Capacity Metrics.

**Siguiente notebook:** NB_05 — AI_TRANSLATE.